In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. 路径导航 (Pathlib 的艺术)
# ==========================================
# 因为 notebook 在 task1_heart_disease/notebook 目录下
# 我们需要连跳两级 parent 才能回到 02_data_cleaning_lab 根目录
root_dir = Path.cwd().parent.parent 

# 定义原始数据和存放结果的路径
raw_data_path = root_dir / "task1_heart_disease" / "raw_data" / "raw_heart_data.csv"
output_dir = root_dir / "task1_heart_disease" / "cleaned_data"

# 读取原始数据
df = pd.read_csv(raw_data_path)

# ==========================================
# 2. 核心清洗步骤
# ==========================================

# --- 任务 A：处理 ID 重复 ---
# 哪怕只有 ID 重复，我们也只保留第一个“见过的”病人，删掉后续可能的错误输入
df.drop_duplicates(subset=['Patient_ID'], keep='first', inplace=True)

# --- 任务 B：性别标准化 (Gender) ---
# 目标：把 Male, male, M, female, F 等统一为 'Male' 和 'Female'
# 方法：先统一变大写并只取首字母(M/F)，再用字典精准映射回标准写法
df['Gender'] = df['Gender'].str.upper().str[0].map({'M': 'Male', 'F': 'Female'})

# --- 任务 C：胆固醇列转型 (Cholesterol) ---
# 这一列混有 '?' 和 'low'，导致变成了字符串。
# errors='coerce' 是精髓：它会把无法转成数字的乱码强行变为空值 (NaN)
df['Cholesterol'] = pd.to_numeric(df['Cholesterol'], errors='coerce')
# 用该列的中位数 (Median) 填补这些空坑，中位数比平均数更能抵抗极端值的影响
df['Cholesterol'] = df['Cholesterol'].fillna(df['Cholesterol'].median())

# --- 任务 D：异常年龄修正 (Age) ---
# 逻辑：找出大于 100 岁或小于 18 岁的行，将它们的 Age 列设为 NaN
df.loc[(df['Age'] > 100) | (df['Age'] < 18), 'Age'] = np.nan
# 用剩下的正常人的“平均年龄”来填补，记得用 round 取整，毕竟年龄没有 45.3 岁一说
avg_age = round(df['Age'].mean())
df['Age'] = df['Age'].fillna(avg_age)

# ==========================================
# 3. 质检与存档
# ==========================================
print("✨ 质检报告：")
print(df.info())  # 确认 Non-Null Count 都是 6，且类型都正确

# 确保 cleaned_data 文件夹存在，不存在就建一个
output_dir.mkdir(exist_ok=True)

# 存盘！
df.to_csv(output_dir / "heart_disease_cleaned_v1.csv", index=False)
print(f"\n✅ 任务 1 完美复原！文件已存至：{output_dir}")

✨ 质检报告：
<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 6
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Patient_ID     6 non-null      int64  
 1   Age            6 non-null      float64
 2   Gender         6 non-null      object 
 3   Cholesterol    6 non-null      float64
 4   Heart_Disease  6 non-null      int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 288.0+ bytes
None

✅ 任务 1 完美复原！文件已存至：c:\projects\machinelearning_study\02_data_cleaning_lab\task1_heart_disease\cleaned_data
